In [1]:
import pandas as pd 
import numpy as np 
import os 

In [2]:
default_path = os.getcwd()
data_path = os.path.join(default_path, '../data')
base_model = os.path.join(default_path, '../base-model')
config_path = os.path.join(default_path, '../config')
log_path = os.path.join(default_path, '../log')
model_path = os.path.join(default_path, '../model')
config_file = "bert-base.json"

In [3]:
reddit = pd.read_csv(os.path.join(data_path, 'aud_dsm.csv'))
reddit

,id,subreddit,author,text,type,time,parent_id,o_id,tok_len
0,akj8u,depression,SarahC,I occasionally look drunk,post,1.262360e+09,akj8u,akj8u,4
1,akj8u,depression,SarahC,"But, in the last 28 months I've kept a high st...",post,1.262360e+09,akj8u,akj8u,22
2,akj8u,depression,SarahC,I've had problems with drive that's possibly m...,post,1.262360e+09,akj8u,akj8u,18
3,akj8u,depression,SarahC,It's been life transforming,post,1.262360e+09,akj8u,akj8u,5
4,akj8u,depression,SarahC,Tramadol works after a few hours and lasts aro...,post,1.262360e+09,akj8u,akj8u,10
...,...,...,...,...,...,...,...,...,...
18269405,t3_eiayp6,depression,BeerxBitches,"The Bridge is not a good documentary, you not ...",comment,1.577837e+09,t3_eiayp6,fconntr,12
18269406,t3_ei96by,depression,noreasontopretend,I was in the same boat as you until I met this...,comment,1.577837e+09,t1_fcokb3z,fconoh8,13
18269407,t3_ei96by,depression,noreasontopretend,It is important you tell these things to the p...,comment,1.577837e+09,t1_fcokb3z,fconoh8,25
18269408,t3_ei96by,depression,noreasontopretend,I only just let my brothers know in the last f...,comment,1.577837e+09,t1_fcokb3z,fconoh8,21


In [4]:
reddit = reddit[reddit.tok_len != 0] 
reddit.reset_index(inplace=True, drop=True)
reddit.duplicated('text').sum()

0

In [5]:
reddit.to_csv(os.path.join(data_path, 'aud_dsm.csv'), index=False)

In [6]:
daily = pd.read_csv(os.path.join(data_path, 'daily_new.csv'))
daily

,text,tok_len
0,"Say , Jim , how about going for a few beers af...",14
1,You know that is tempting but is really not go...,14
2,What do you mean ? It will help us to relax .,12
3,Do you really think so ? I don't . It will jus...,24
4,I guess you are right.But what shall we do ? I...,19
...,...,...
100251,are you kidding ? Can you afford it ? Do you t...,22
100252,"never mind that , I'll take care of it . Are y...",17
100253,"yeah , I think so .",6
100254,ok . I'll make the arrangements . It will be g...,13


In [9]:
reddit_samp = reddit.sample(100000)
reddit_samp.reset_index(inplace=True, drop=True)

In [11]:
reddit_doc = reddit_samp.text.tolist()

In [12]:
from sentence_transformers import SentenceTransformer, util
model = SentenceTransformer('paraphrase-distilroberta-base-v1')

embeddings = model.encode(reddit_doc, convert_to_tensor=True)

In [37]:
print(np.shape(embeddings))
embeddings

torch.Size([100000, 768])


tensor([[-0.1791,  0.3885,  0.4159,  ..., -0.1798,  0.3356, -0.0184],
        [-0.2079,  0.1226,  0.0672,  ..., -0.1313, -0.0892,  0.3501],
        [ 0.1282, -0.1631, -0.2264,  ..., -0.0791, -0.1038,  0.0493],
        ...,
        [-0.0219,  0.6735,  0.3889,  ...,  0.1283,  0.1631,  0.0331],
        [-0.1999,  0.3665,  0.1724,  ..., -0.0908, -0.1269, -0.0182],
        [-0.2457,  0.4567,  0.0862,  ..., -0.0980, -0.0643, -0.3009]])

In [10]:
daily_doc = daily.text.tolist()

In [118]:
text = daily_doc[88103]
text

"I'll spent more time with you in the future , Dina ."

In [119]:
text_emb = model.encode(text, convert_to_tensor=True)

In [120]:
len(embeddings), len(embeddings[0])

(100000, 768)

In [121]:
cosine_scores = util.pytorch_cos_sim(embeddings, text_emb).tolist()
cosine_scores = [score[0] for score in cosine_scores]
len(cosine_scores)

100000

In [122]:
idx_list = list(range(len(cosine_scores)))
cosine_dict = dict(zip(idx_list, cosine_scores))

In [123]:
sorted(cosine_dict.items(), key=lambda x: x[1], reverse=True)[:3]

[(20038, 0.38692188262939453),
 (64362, 0.3690432906150818),
 (93856, 0.3644183576107025)]

In [124]:
text

"I'll spent more time with you in the future , Dina ."

In [127]:
reddit_doc[20038]

'I wish I had more time with her'

In [126]:
def get_idx(
    text, model, embeddings):
    text_emb = model.encode(text, convert_to_tensor=True)
    cosine_scores = util.pytorch_cos_sim(embeddings, text_emb).tolist()
    cosine_scores = [score[0] for score in cosine_scores]
    idx_list = list(range(len(cosine_scores)))
    
    cosine_dict = dict(zip(idx_list, cosine_scores))
    cosine_dict = sorted(cosine_dict.items(), key=lambda x: x[1], reverse=True)
    
    reddit_idx = [] 
    # print(cosine_dict)
    for idx, value in cosine_dict:
        if value > 0.7:
            reddit_idx.append(idx)
        if value < 0.7:
            break
    return reddit_idx

In [23]:
get_idx(text, model, embeddings)

[]

In [24]:
text

"I can , but not as well as I'd like to . I'll be studying Chinese until the day I die !"